Para essa sprint precisei terminar de fazer algumas limpezas que não havia conseguido na sprint passada.

Depois dos dados limpos, optei por fazer um dataset consolidado, pois nesse caso são poucos dados e tabelas. A partir desse ponto decidi como separar a tabela consolidada em dimensões.

 Depois mais a diante eu criei algumas views. Mas primeiro relembrei as minhas perguntas:

- Qual é a tendência das notas dos filmes e séries em relação à duração ao longo da década?
- Na década análisada, qual foi a média de nota do gênero?
- Qual ator e atriz mais bem avaliados até a coleta dos dados?
- Qual ator e qual atriz foi mais requisitado(a) nessa década, e qual a média das notas que das produções que ele(a) atuou?
- Podemos levantar quais atores e atrizes trabalharam tanto em séries quanto em filmes, e quantos trabalharam exclusivamente em filmes e exclusivamente em séries.


Como tive muita dificuldade na sprint passada fiquei com receio de não conseguir trabalhar com todos os dados, a minha intenção era usar o dataframe com series e filmes, e depois tbem trababar com duração dos mesmos e a coluna de profissão,, além disso diminui os dados somente para filmes de 2000 até agora.

Acabei retirando essas informações da análise pelo motivo já explicado. Agora que entendi como o Glue funciona acho que não tinha necessidade de tirar as informações, mas ainda sim acho que foi uma boa decisão, melhor apresentar algo menor e bem feito do que algo grande e mal feito.

Então com base nos dados que eu refinei as perguntas adaptadas daquelas iniciais seriam:

- Qual a quantidade de filmes realizados em cada ano? e Qual a média de nota em cada ano?
- Quais são os artistas mais bem avaliados dentro de sua categoria de idade, seu filme mais bem avaliado e a nota?
- Qual a quantidade de filmes cada categoria de idade participou, quantos artistas envolvidos e a media de notas dos filmes em relação aos artistas?
- Para os artistas, qual categoria ele pertence, qual sua avaliação, quantos filmes ele atuou e a média de notas dos filmes que participou?
- Qual a quantidade de filmes nas categorias 'War' e 'Crime' e qual a media de notas dessas categorias?
- Quais são os top 5 dentro da categoria 'War' e 'Crime', suas notas, qual artista mais bem avaliado?

Tentei manter o máximo que pude manter as perguntas que havia selecionado, e tentei encontrar novas perguntas com base no que encontrei.

In [1]:
# instalando as bibliotecas necessárias
!pip install -q pyspark
!pip install -q findspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# Importando as bibliotecas
import matplotlib.pyplot as plt
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark import SparkContext, SQLContext
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import random
from pyspark.sql.functions import (col,
                                   rand,
                                   when,
                                   udf,
                                   monotonically_increasing_id,
                                   concat,
                                   broadcast,
                                   explode,
                                   array,
                                   coalesce,
                                   lit)

## Iniciando  a criação de uma tabela consolidade

In [3]:
# iniciando sessão Spark
spark = SparkSession.builder.appName("mySparkSession").getOrCreate()

In [ ]:
# Baixando o arquivo e lendo as primeiras linhas
df_part_1 = spark.read.parquet("/content/run-1721586637551-part-block-0-r-00000-snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_2 = spark.read.parquet("/content/run-1721586637551-part-block-0-r-00001-snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_3 = spark.read.parquet("/content/run-1721586637551-part-block-0-r-00002-snappy.parquet", header= True, inferSchema=True, sep="|")
df_artistas = spark.read.parquet("/content/run-1721255864609-part-block-0-r-00000-snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_1.show(5)
df_part_2.show(5)
df_part_3.show(5)
df_artistas.show(5)

+-----------+-----------+--------------------+--------------------+-------------+-------------+--------------+--------------------+-----------+-------------+-------------+---------+---------+
|numerovotos|partition_1|              genero|      titulopincipal|generoartista|   personagem|   nomeartista|      titulooriginal|partition_0|anonascimento|anolancamento|notamedia|       id|
+-----------+-----------+--------------------+--------------------+-------------+-------------+--------------+--------------------+-----------+-------------+-------------+---------+---------+
|      85400|         09|Comedy,Fantasy,Ro...|      Kate & Leopold|      actress|   Kate McKay|      Meg Ryan|      Kate & Leopold|         06|         1961|         2001|      6.4|tt0035423|
|      85400|         09|Comedy,Fantasy,Ro...|      Kate & Leopold|        actor|Charlie McKay| Breckin Meyer|      Kate & Leopold|         06|         1974|         2001|      6.4|tt0035423|
|      85400|         09|Comedy,Fantasy,

In [ ]:
df = df_part_1.unionByName(df_part_2, allowMissingColumns=True)
df = df.unionByName(df_part_3, allowMissingColumns=True)
df.show(5)

+-----------+-----------+--------------------+--------------------+-------------+-------------+--------------+--------------------+-----------+-------------+-------------+---------+---------+
|numerovotos|partition_1|              genero|      titulopincipal|generoartista|   personagem|   nomeartista|      titulooriginal|partition_0|anonascimento|anolancamento|notamedia|       id|
+-----------+-----------+--------------------+--------------------+-------------+-------------+--------------+--------------------+-----------+-------------+-------------+---------+---------+
|      85400|         09|Comedy,Fantasy,Ro...|      Kate & Leopold|      actress|   Kate McKay|      Meg Ryan|      Kate & Leopold|         06|         1961|         2001|      6.4|tt0035423|
|      85400|         09|Comedy,Fantasy,Ro...|      Kate & Leopold|        actor|Charlie McKay| Breckin Meyer|      Kate & Leopold|         06|         1974|         2001|      6.4|tt0035423|
|      85400|         09|Comedy,Fantasy,

## Código para usar no script para terminar a limpeza dos dados

In [ ]:
# Casting do atributo id e gender para ficar no mesmo esquema que os DFs anteriores
df = df.withColumn('notamedia', df['notamedia'].cast('int'))
df = df.withColumn('anonascimento', df['anonascimento'].cast('int'))
df = df.withColumn('anolancamento', df['anolancamento'].cast('int'))

df.printSchema()

# Eliminando colunas
df = df.drop("partition_0", "partition_1")

# Renomear colunas específicas
df = df.withColumnRenamed("genero", "generofilme").withColumnRenamed('id', "idfilme")
df.show(5)


# Selecionando as linhas com pelo menos um dos valores "war" ou "crime"
df = df.filter((col("generofilme").contains("War") | col("generofilme").contains("Crime")))
df = df[df["anoNascimento"] >= 1944]

# Modificando o valor da coluna 'generoArtista' para valores numericos
df = df.withColumn("generoartista",
                    when(df["generoartista"] == "actor", 2).otherwise(1))
df.show(5)

# Criando uma coluna para idade
df_filtered = df.withColumn("idadeartista", 2024 - df["anoNascimento"])

# Criando a coluna de categoria de idade
df_filtered = df_filtered.withColumn("categoriaIdade",
                                     when((df_filtered["idadeartista"] >= 1) & (df_filtered["idadeartista"] <= 20), "juvenil")
                                     .when((df_filtered["idadeartista"] >= 21) & (df_filtered["idadeartista"] <= 40), "adultos")
                                     .when((df_filtered["idadeartista"] >= 41) & (df_filtered["idadeartista"] <= 60), "maduros")
                                     .otherwise("idosos"))

# Exiba o DataFrame resultante
df_filtered.show(5)


# Dividindo a coluna "generofilme" em três colunas
df_split = df_filtered.withColumn("genero_lista", split(df_filtered["generofilme"], ","))

# Extracting elements from the array column
df_split = df_split.withColumn("generofilme1", df_split["genero_lista"].getItem(0)) \
                   .withColumn("generofilme2", df_split["genero_lista"].getItem(1)) \
                   .withColumn("generofilme3", df_split["genero_lista"].getItem(2))



# Substitui os valores nulos nas colunas generofime2 e generofilme3
df_split = df_split.withColumn("generofilme2", when(df_split["generofilme2"].isNull(), "sem informacao").otherwise(df_split["generofilme2"])).withColumn("generofilme3", when(df_split["generofilme3"].isNull(), "sem informacao").otherwise(df_split["generofilme3"]))


# Eliminando "genero" e "genero_lista"
df_split = df_split.drop("generofilme", "genero_lista")

# Exibindo o resultado
df_split.show()


# Inner join para ter os dados que quero
df_joined = df_split.join(df_artistas, on=df_split["nomeartista"] == df_artistas["nome"], how="inner")

# Renomear colunas específicas
df_joined = df_joined.withColumnRenamed("id", "idartista").withColumnRenamed('nome', "nomeartistaavalicao").withColumnRenamed('nomeOriginal', "nomeoriginalavalicao").withColumnRenamed('genero', "generoartistaavaliacao").withColumnRenamed('avaliacao', "avaliacaoartista")
df_joined.show()


# Criando uma nova coluna "id_artista" com valores únicos
df_joined = df_joined.withColumn("idpersonagem", monotonically_increasing_id())


# Colocando as colunas na ordem desejada
df_ordenado = df_joined.select('idfilme',
                             'titulopincipal',
                             'titulooriginal',
                             'anolancamento',
                             'numerovotos',
                             'notamedia',
                             'generofilme1',
                             'generofilme2',
                             'generofilme3',
                             'nomeartista',
                             'generoartista',
                            'idpersonagem',
                             'personagem',
                             'idartista',
                             'avaliacaoartista',
                             'anonascimento',
                             'idadeartista',
                             'categoriaIdade',
                             'nomeartistaavalicao',
                             'generoartistaavaliacao',
                             'nomeoriginalavalicao',
)
df_ordenado.show(120)

root
 |-- numerovotos: long (nullable = true)
 |-- partition_1: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- titulopincipal: string (nullable = true)
 |-- generoartista: string (nullable = true)
 |-- personagem: string (nullable = true)
 |-- nomeartista: string (nullable = true)
 |-- titulooriginal: string (nullable = true)
 |-- partition_0: string (nullable = true)
 |-- anonascimento: integer (nullable = true)
 |-- anolancamento: integer (nullable = true)
 |-- notamedia: integer (nullable = true)
 |-- id: string (nullable = true)

+-----------+--------------------+--------------------+-------------+-------------+--------------+--------------------+-------------+-------------+---------+---------+
|numerovotos|         generofilme|      titulopincipal|generoartista|   personagem|   nomeartista|      titulooriginal|anonascimento|anolancamento|notamedia|  idfilme|
+-----------+--------------------+--------------------+-------------+-------------+--------------+-----

## Criando a estrutura para a modelagem dos dados com a tabela refined

In [4]:
# Baixando o arquivo e lendo as primeiras linhas
df_part_1 = spark.read.parquet("/content/part-00000-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_2 = spark.read.parquet("/content/part-00001-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_3 = spark.read.parquet("/content/part-00004-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_4 = spark.read.parquet("/content/part-00006-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_5 = spark.read.parquet("/content/part-00007-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_6 = spark.read.parquet("/content/part-00008-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_7 = spark.read.parquet("/content/part-00009-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_8 = spark.read.parquet("/content/part-00010-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_9 = spark.read.parquet("/content/part-00011-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_10 = spark.read.parquet("/content/part-00012-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_11 = spark.read.parquet("/content/part-00013-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_12 = spark.read.parquet("/content/part-00014-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_13 = spark.read.parquet("/content/part-00015-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_14 = spark.read.parquet("/content/part-00016-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_15 = spark.read.parquet("/content/part-00017-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")
df_part_16 = spark.read.parquet("/content/part-00018-72a19eee-3d28-42f0-b70f-25d171f0d757-c000.snappy.parquet", header= True, inferSchema=True, sep="|")

df = df_part_1.unionByName(df_part_2, allowMissingColumns=True)
df = df.unionByName(df_part_3, allowMissingColumns=True)
df = df.unionByName(df_part_4, allowMissingColumns=True)
df = df.unionByName(df_part_5, allowMissingColumns=True)
df = df.unionByName(df_part_6, allowMissingColumns=True)
df = df.unionByName(df_part_7, allowMissingColumns=True)
df = df.unionByName(df_part_8, allowMissingColumns=True)
df = df.unionByName(df_part_9, allowMissingColumns=True)
df = df.unionByName(df_part_10, allowMissingColumns=True)
df = df.unionByName(df_part_11, allowMissingColumns=True)
df = df.unionByName(df_part_12, allowMissingColumns=True)
df = df.unionByName(df_part_13, allowMissingColumns=True)
df = df.unionByName(df_part_14, allowMissingColumns=True)
df = df.unionByName(df_part_15, allowMissingColumns=True)
df = df.unionByName(df_part_16, allowMissingColumns=True)

df = df.dropDuplicates()
df.show(5)
df.count()

+----------+------------------+--------------------+-------------+-----------+---------+------------+------------+------------+------------+-------------+------------+-------------+---------+----------------+-------------+------------+--------------+--------------------+----------------------+---------------------+
|   idfilme|    titulopincipal|      titulooriginal|anolancamento|numerovotos|notamedia|generofilme1|generofilme2|generofilme3| nomeartista|generoartista|idpersonagem|   personagem|idartista|avaliacaoartista|anonascimento|idadeartista|categoriaIdade|nomeartistaavaliacao|generoartistaavaliacao|nomeoriginalavaliacao|
+----------+------------------+--------------------+-------------+-----------+---------+------------+------------+------------+------------+-------------+------------+-------------+---------+----------------+-------------+------------+--------------+--------------------+----------------------+---------------------+
| tt4619940|Chronicles of Evil|   Ak-ui yeon-dae-

120

## Criando as tabelas

### Tabela `categoriaidade`

In [5]:
# Obtendo os valores únicos da coluna 'categoriaidade'
valores_unicos = df.select("categoriaidade").distinct()

# Mapeiando os IDs para as categorias correspondentes
regra_ids = {
    "juvenil": 1,
    "adultos": 2,
    "maduros": 3,
    "idosos": 4
}

# Crie uma nova tabela com os IDs e os nomes das categorias
categoriaidade = valores_unicos.withColumn(
    "idcategoriaidade",
    when(valores_unicos["categoriaidade"] == "juvenil", 1)
    .when(valores_unicos["categoriaidade"] == "adultos", 2)
    .when(valores_unicos["categoriaidade"] == "maduros", 3)
    .when(valores_unicos["categoriaidade"] == "idosos", 4)
    .otherwise(None)  # Adiciona um valor padrão se necessário
)

categoriaidade = categoriaidade.select("idcategoriaidade", "categoriaidade")

categoriaidade.show()

+----------------+--------------+
|idcategoriaidade|categoriaidade|
+----------------+--------------+
|               3|       maduros|
|               2|       adultos|
|               4|        idosos|
|               1|       juvenil|
+----------------+--------------+



### Tabela `artista`

In [6]:
# Obtendo os valores únicos da coluna 'categoriaidade'
valores_unicos = categoriaidade.select("idcategoriaidade", "categoriaidade").distinct()

# Em seguida, junte os DataFrames 'df' e 'valores_unicos' para obter os IDs correspondentes
artista = df.join(valores_unicos, on="categoriaidade", how="inner")

# Selecione as colunas desejadas
artista = artista.select("idartista", "idcategoriaidade", "nomeartista", "generoartista",
                         "nomeoriginalavaliacao", "anonascimento",
                         "idadeartista", "avaliacaoartista" ).distinct()

# Renomeia a coluna nomeoriginalavaliacao para nomeoriginal
artista = artista.withColumnRenamed("nomeoriginalavaliacao", "nomeoriginal")

# Exiba o resultado
artista.show(5)

+---------+----------------+--------------------+-------------+--------------------+-------------+------------+----------------+
|idartista|idcategoriaidade|         nomeartista|generoartista|        nomeoriginal|anonascimento|idadeartista|avaliacaoartista|
+---------+----------------+--------------------+-------------+--------------------+-------------+------------+----------------+
|     6162|               3|        Paul Bettany|            2|        Paul Bettany|         1971|          53|         111.077|
|    24045|               3|Joseph Gordon-Levitt|            2|Joseph Gordon-Levitt|         1981|          43|         127.441|
|  1024395|               3|        Ma Dong-seok|            2|              마동석|         1971|          53|         116.992|
|   115440|               2|      Sydney Sweeney|            1|      Sydney Sweeney|         1997|          27|          106.99|
|   125025|               2|           Joey King|            1|           Joey King|         1999|  

### Tabela `personagem`

In [7]:
personagem = df.select(
    'idpersonagem', 'personagem'
).distinct()

personagem.show(5)
personagem.count()

+------------+------------+
|idpersonagem|  personagem|
+------------+------------+
|           6|     Jang-su|
|           8|Detective Ma|
|          13|Detective Oh|
|           4|   Hyung-Suk|
|          14|   Sang-baek|
+------------+------------+
only showing top 5 rows



120

### Tabela `elencopersonagem`

In [8]:
# Selecionar as colunas relevantes e adicionar IDs
elencopersonagem = df.select(
    'idpersonagem', 'idartista'
).distinct()

# Adicionar uma coluna de IDs usando monotonically_increasing_id
elencopersonagem = elencopersonagem.withColumn('idelenco', monotonically_increasing_id())

elencopersonagem = elencopersonagem.select('idelenco', 'idpersonagem', 'idartista')

# Mostrar o resultado
elencopersonagem.show(5)

+--------+------------+---------+
|idelenco|idpersonagem|idartista|
+--------+------------+---------+
|       0|           9|  1024395|
|       1|           1|  1024395|
|       2|           6|  1024395|
|       3|           3|  1024395|
|       4|          11|  1024395|
+--------+------------+---------+
only showing top 5 rows



### Tabela `genero`

In [9]:
# Dicionário de mapeamento dos IDs para os gêneros
genero_ids = {
    "sem informacao": 0,
    "Crime": 1,
    "Drama": 2,
    "Documentary": 3,
    "Biography": 4,
    "Comedy": 5,
    "Action": 6,
    "Thriller": 7,
    "Adventure": 8,
    "War": 9,
    "Mystery": 10,
    "Horror": 11,
    "Romance": 12,
    "Fantasy": 13,
    "Sci-Fi": 14
}

# Função UDF para mapear o gênero para o ID correspondente
def genero_para_id(genero):
    return genero_ids.get(genero, None)

genero_para_id_udf = udf(genero_para_id, IntegerType())

# Obtendo os valores únicos das colunas
categorias1 = df.select("generofilme1").distinct().withColumnRenamed("generofilme1", "generofilme")
categorias2 = df.select("generofilme2").distinct().withColumnRenamed("generofilme2", "generofilme")
categorias3 = df.select("generofilme3").distinct().withColumnRenamed("generofilme3", "generofilme")

# Combinando as categorias únicas das três colunas
categorias_gerais = categorias1.union(categorias2).union(categorias3).distinct()

# Adicionando a coluna de IDs utilizando a função UDF
genero = categorias_gerais.withColumn("idgenero", genero_para_id_udf("generofilme"))

# Selecionando as colunas 'idgenero' e 'generofilme'
genero = genero.select("idgenero", "generofilme")

genero.show(20)

+--------+--------------+
|idgenero|   generofilme|
+--------+--------------+
|       1|         Crime|
|       2|         Drama|
|       3|   Documentary|
|       4|     Biography|
|       5|        Comedy|
|       6|        Action|
|       7|      Thriller|
|       8|     Adventure|
|       9|           War|
|      10|       Mystery|
|      11|        Horror|
|      12|       Romance|
|      13|       Fantasy|
|       0|sem informacao|
|      14|        Sci-Fi|
+--------+--------------+



### Tabela `filmegenero`

In [10]:
# # Criar tabelas temporárias
# df.createOrReplaceTempView("df")
# genero.createOrReplaceTempView("genero")

# # Consulta SQL para obter o idgenero para generofilme1
# sql_query1 = """
# SELECT f.idfilme, g.idgenero
# FROM df f
# JOIN genero g ON f.generofilme1 = g.generofilme
# """

# # Executar a consulta SQL
# filmegenero1 = spark.sql(sql_query1)

# # Consulta SQL para obter o idgenero para generofilme2
# sql_query2 = """
# SELECT f.idfilme, g.idgenero
# FROM df f
# JOIN genero g ON f.generofilme2 = g.generofilme
# """

# # Executar a consulta SQL
# filmegenero2 = spark.sql(sql_query2)

# # Consulta SQL para obter o idgenero para generofilme3
# sql_query3 = """
# SELECT f.idfilme, g.idgenero
# FROM df f
# JOIN genero g ON f.generofilme3 = g.generofilme
# """

# # Executar a consulta SQL
# filmegenero3 = spark.sql(sql_query3)

# # Unir os resultados dos três selects
# filmegenero = filmegenero1.union(filmegenero2).union(filmegenero3)

# # Criar ou substituir a tabela temporária 'filmegenero'
# filmegenero.createOrReplaceTempView("filmegenero")

# # Remover linhas duplicadas
# filmegenero = filmegenero.dropDuplicates()

# # Mostrar o resultado
# filmegenero.show(5)

In [11]:
# obter os generos das três colunas do df
filmegenero1 = df.join(genero, df.generofilme1 == genero.generofilme) \
                 .select(df.idfilme, genero.idgenero)

filmegenero2 = df.join(genero, df.generofilme2 == genero.generofilme) \
                 .select(df.idfilme, genero.idgenero)

filmegenero3 = df.join(genero, df.generofilme3 == genero.generofilme) \
                 .select(df.idfilme, genero.idgenero)

# Unir os resultados dos três DataFrames
filmegenero = filmegenero1.union(filmegenero2).union(filmegenero3)

# Remover linhas duplicadas
filmegenero = filmegenero.dropDuplicates()
filmegenero.show(5)

+---------+--------+
|  idfilme|idgenero|
+---------+--------+
|tt1893256|       6|
|tt4106514|       1|
|tt0285823|       6|
|tt1935194|       6|
|tt2082221|       1|
+---------+--------+
only showing top 5 rows



### Tabela `filme`

In [12]:
# # Criar uma tabela temporária para ElencoPersonagens
# elencopersonagem.createOrReplaceTempView("elencopersonagem")

# # Criar uma tabela temporária para df
# df.createOrReplaceTempView("df")

# sql_query = """
# SELECT f.idfilme, f.titulopincipal, f.titulooriginal, f.anolancamento, f.numerovotos, f.notamedia, ep.idelenco
# FROM df f
# LEFT JOIN elencopersonagem ep ON f.idpersonagem = ep.idpersonagem
# """

# # Executar a consulta SQL
# filme = spark.sql(sql_query)

# # Mostrar o resultado
# filme.show(5)

In [13]:
# Realizar o join diretamente com DataFrames
filme = df.join(elencopersonagem, df.idpersonagem == elencopersonagem.idpersonagem, how="left") \
    .select(
        df.idfilme,
        df.titulopincipal,
        df.titulooriginal,
        df.anolancamento,
        df.numerovotos,
        df.notamedia,
        elencopersonagem.idelenco
    )

filme = filme.select("idfilme", "idelenco", "titulopincipal", "titulooriginal",
                     "anolancamento", "numerovotos", "notamedia")
filme.show(5)

+----------+--------+------------------+--------------------+-------------+-----------+---------+
|   idfilme|idelenco|    titulopincipal|      titulooriginal|anolancamento|numerovotos|notamedia|
+----------+--------+------------------+--------------------+-------------+-----------+---------+
| tt4619940|       6|Chronicles of Evil|   Ak-ui yeon-dae-gi|         2015|       1998|        6|
| tt3696126|      10|        One on One|             Ildaeil|         2014|       1029|        5|
| tt1560154|      13|  Insadong Scandal|Insadong seukaendeul|         2009|        190|        6|
| tt9174334|       3|     The Villagers|     Dongnesaramdeul|         2018|        999|        6|
|tt15838850|      11|       The Roundup|       Beomjoidosi 2|         2022|       4921|        7|
+----------+--------+------------------+--------------------+-------------+-----------+---------+
only showing top 5 rows



### Testes

In [14]:
filme.createOrReplaceTempView("filme")
artista.createOrReplaceTempView("artista")
categoriaidade.createOrReplaceTempView("categoriaidade")
personagem.createOrReplaceTempView("personagem")
genero.createOrReplaceTempView("genero")
filmegenero.createOrReplaceTempView("filmegenero")

In [15]:
# teste1 = spark.sql("""
#     SELECT *
#     FROM filme
# """)

# teste1.show()

In [16]:
# teste2 = spark.sql("""
# SELECT f.idfilme, f.titulopincipal, f.titulooriginal, f.anolancamento, f.numerovotos, f.notamedia, a.nomeartista
# FROM filme f
# JOIN elencopersonagem ep ON f.idelenco = ep.idelenco
# JOIN artista a ON ep.idartista = a.idartista -- Changed 'f.idartista' to 'ep.idartista'
# """)

# teste2.show()

In [17]:
# # Executar a consulta SQL corrigida
# teste3 = spark.sql("""
# SELECT p.personagem AS NomePersonagem,
#        a.nomeartista AS NomeArtista,
#        f.titulopincipal AS TituloPrincipal
# FROM filme f
# JOIN elencopersonagem ep ON f.idelenco = ep.idelenco
# JOIN artista a ON ep.idartista = a.idartista
# JOIN personagem p ON ep.idpersonagem = p.idpersonagem
# WHERE f.idfilme = "tt2082221";
# """)

# # Mostrar o resultado
# teste3.show()

In [18]:
# # Register the categoriaidade DataFrame as a temporary view
# teste4 = spark.sql("""
# SELECT a.idartista, a.nomeartista, ci.categoriaIdade
# FROM artista a
# JOIN categoriaidade ci ON a.idcategoriaidade = ci.idcategoriaidade
# """)

# teste4.show(25)

In [19]:
# teste5 = spark.sql("""
# SELECT f.*,
#        p.personagem AS NomePersonagem,
#        a.nomeartista AS NomeArtista,
#        a.avaliacaoartista AS AvaliacaoArtista
# FROM filme f
# JOIN elencopersonagem ep ON f.idelenco = ep.idelenco
# JOIN artista a ON ep.idartista = a.idartista
# JOIN personagem p ON ep.idpersonagem = p.idpersonagem
# """)

# teste5.show()

## Criando as dimensões

In [20]:
# Supondo que df seja seu DataFrame com dados da tabela Filme
dimfilme = df.select(
    'idfilme', 'titulopincipal', 'titulooriginal', 'anolancamento',
    'numerovotos', 'notamedia'
).distinct()
dimfilme.createOrReplaceTempView("dimfilme")

# Unir as tabelas com aliases
dimelencopersonagem = elencopersonagem.alias("ep") \
    .join(artista.alias("a"), col("ep.idartista") == col("a.idartista"), "left") \
    .join(categoriaidade.alias("c"), col("c.idcategoriaidade") == col("a.idcategoriaidade"), "left") \
    .join(personagem.alias("p"), col("ep.idpersonagem") == col("p.idpersonagem"), "left") \
    .select(
        col("ep.idelenco"),
        col("ep.idpersonagem"),
        col("ep.idartista"),
        col("a.nomeartista"),
        col("a.nomeoriginal"),
        col("a.idadeartista"),
        col("a.generoartista"),
        col("a.anonascimento"),
        col("a.avaliacaoartista"),
        col("p.personagem"),
        col('c.idcategoriaidade'),
        col('c.categoriaIdade')
    ) \
    .distinct()
dimelencopersonagem.createOrReplaceTempView("dimelencopersonagem")

dimgenero = filmegenero.alias('fg') \
    .join(
        genero.alias('g'),
        coalesce(col('fg.idgenero'), lit(-1)) == coalesce(col('g.idgenero'), lit(-1)),
        'inner'
    ).select(
        col('fg.idgenero'),
        col('g.generofilme'),
        col('fg.idfilme')
    ).distinct()
dimgenero.createOrReplaceTempView("dimgenero")

dimfilme.show(5)
dimelencopersonagem.show(5)
dimgenero.show(5)

+---------+----------------+--------------------+-------------+-----------+---------+
|  idfilme|  titulopincipal|      titulooriginal|anolancamento|numerovotos|notamedia|
+---------+----------------+--------------------+-------------+-----------+---------+
|tt1825955|     Midnight FM|         Simya-ui FM|         2010|       2739|        6|
|tt9225192|     Unstoppable|    Seongnan hwangso|         2018|       3450|        6|
|tt1560154|Insadong Scandal|Insadong seukaendeul|         2009|        190|        6|
|tt7468056|     The Outlaws|         Beomjoidosi|         2017|       8730|        7|
|tt6173644|        Derailed|           Doo namja|         2016|        692|        6|
+---------+----------------+--------------------+-------------+-----------+---------+
only showing top 5 rows

+--------+------------+---------+---------------+---------------+------------+-------------+-------------+----------------+--------------+----------------+--------------+
|idelenco|idpersonagem|idartis

In [21]:
df.createOrReplaceTempView("df")

# Unir as tabelas para criar a tabela fato
fatofilme = filme.alias("f") \
    .join(dimfilme.alias("d"), col("f.idfilme") == col("d.idfilme"), "left") \
    .join(dimelencopersonagem.alias("e"), col("f.idelenco") == col("e.idelenco"), "left") \
    .join(dimgenero.alias("g"), col("f.idfilme") == col("g.idfilme"), "left") \
    .select(
        col("d.idfilme"),
        col("e.idelenco"),
        col("g.idgenero"),
    ) \
    .distinct()

# Criar ou substituir a tabela temporária para a tabela fato
fatofilme.createOrReplaceTempView("fatofilme")

# Mostrar o resultado
fatofilme.show(5)

+---------+--------+--------+
|  idfilme|idelenco|idgenero|
+---------+--------+--------+
|tt1219827|      25|       1|
|tt7468056|       9|       7|
|tt1640548|      82|       2|
|tt0936501|     119|       6|
|tt1560154|      13|       1|
+---------+--------+--------+
only showing top 5 rows



## Codigo para o script

In [ ]:
import sys
from awsglue.transforms import *
from pyspark.sql.functions import col
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame
import re

# Argumentos do Job
args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

# Caminhos de saída
output_path = 's3://pamela-desafio-06/data-lake-pamela/Refined/modelagem/'
dimensoes_output_path = 's3://pamela-desafio-06/data-lake-pamela/Refined/dimensoes/'

# Função para salvar no Glue Catalog
def save_to_glue_catalog(df, output_path, database, table_name):
    dynamic_frame = DynamicFrame.fromDF(df, glueContext, table_name)
    sink = glueContext.getSink(connection_type="s3", path=output_path,
                               enableUpdateCatalog=True, updateBehavior="UPDATE_IN_DATABASE",
                               partitionKeys=[], compression="snappy")
    sink.setFormat("glueparquet")
    sink.setCatalogInfo(catalogDatabase=database, catalogTableName=table_name)
    sink.writeFrame(dynamic_frame)

# Leitura do DataFrame
df = glueContext.create_dynamic_frame.from_catalog(
    database="database-desafio",
    table_name="job_refined"
).toDF()

# tabela categoriaidade
# Obtendo os valores únicos da coluna 'categoriaidade'
valores_unicos = df.select("categoriaidade").distinct()

# Mapeiando os IDs para as categorias correspondentes
regra_ids = {
    "juvenil": 1,
    "adultos": 2,
    "maduros": 3,
    "idosos": 4
}

# Crie uma nova tabela com os IDs e os nomes das categorias
categoriaidade = valores_unicos.withColumn(
    "idcategoriaidade",
    when(valores_unicos["categoriaidade"] == "juvenil", 1)
    .when(valores_unicos["categoriaidade"] == "adultos", 2)
    .when(valores_unicos["categoriaidade"] == "maduros", 3)
    .when(valores_unicos["categoriaidade"] == "idosos", 4)
    .otherwise(None)
)

categoriaidade = categoriaidade.select("idcategoriaidade", "categoriaidade")
categoriaidade.show(5)

# tabela artista
# Obtendo os valores únicos da coluna 'categoriaidade'
valores_unicos = categoriaidade.select("idcategoriaidade", "categoriaidade").distinct()
artista = df.join(valores_unicos, on="categoriaidade", how="inner")

# Selecione as colunas desejadas
artista = artista.select("idartista", "idcategoriaidade", "nomeartista", "generoartista",
                         "nomeoriginalavaliacao", "anonascimento",
                         "idadeartista", "avaliacaoartista" ).distinct()

# Renomeia a coluna nomeoriginalavaliacao para nomeoriginal
artista = artista.withColumnRenamed("nomeoriginalavaliacao", "nomeoriginal")
artista.show(5)

# tabela personagem
personagem = df.select(
    'idpersonagem', 'personagem'
).distinct()
personagem.show(5)

# tabela elencopersonagem
# Selecionar as colunas relevantes e adicionar IDs
elencopersonagem = df.select(
    'idpersonagem', 'idartista'
).distinct()

# Adicionar uma coluna de IDs usando monotonically_increasing_id
elencopersonagem = elencopersonagem.withColumn('idelenco', monotonically_increasing_id())
elencopersonagem = elencopersonagem.select('idelenco', 'idpersonagem', 'idartista')
elencopersonagem.show(5)

# tabela genero
# Dicionário de mapeamento dos IDs para os gêneros
genero_ids = {
    "sem informacao": 0,
    "Crime": 1,
    "Drama": 2,
    "Documentary": 3,
    "Biography": 4,
    "Comedy": 5,
    "Action": 6,
    "Thriller": 7,
    "Adventure": 8,
    "War": 9,
    "Mystery": 10,
    "Horror": 11,
    "Romance": 12,
    "Fantasy": 13,
    "Sci-Fi": 14
}

# Função UDF para mapear o gênero para o ID correspondente
def genero_para_id(genero):
    return genero_ids.get(genero, None)

genero_para_id_udf = udf(genero_para_id, IntegerType())

# Obtendo os valores únicos das colunas
categorias1 = df.select("generofilme1").distinct().withColumnRenamed("generofilme1", "generofilme")
categorias2 = df.select("generofilme2").distinct().withColumnRenamed("generofilme2", "generofilme")
categorias3 = df.select("generofilme3").distinct().withColumnRenamed("generofilme3", "generofilme")

# Combinando as categorias únicas das três colunas
categorias_gerais = categorias1.union(categorias2).union(categorias3).distinct()

# Adicionando a coluna de IDs utilizando a função UDF
genero = categorias_gerais.withColumn("idgenero", genero_para_id_udf("generofilme"))

# Selecionando as colunas 'idgenero' e 'generofilme'
genero = genero.select("idgenero", "generofilme")
genero.show(20)

# tabela filmegenero
# obter os generos das três colunas do df
filmegenero1 = df.join(genero, df.generofilme1 == genero.generofilme) \
                 .select(df.idfilme, genero.idgenero)

filmegenero2 = df.join(genero, df.generofilme2 == genero.generofilme) \
                 .select(df.idfilme, genero.idgenero)

filmegenero3 = df.join(genero, df.generofilme3 == genero.generofilme) \
                 .select(df.idfilme, genero.idgenero)

# Unir os resultados dos três DataFrames
filmegenero = filmegenero1.union(filmegenero2).union(filmegenero3)

# Remover linhas duplicadas
filmegenero = filmegenero.dropDuplicates()
filmegenero.show(5)

# Tabela filme
# Realizar o join diretamente com DataFrames
filme = df.join(elencopersonagem, df.idpersonagem == elencopersonagem.idpersonagem, how="left") \
    .select(
        df.idfilme,
        df.titulopincipal,
        df.titulooriginal,
        df.anolancamento,
        df.numerovotos,
        df.notamedia,
        elencopersonagem.idelenco
    )

filme = filme.select("idfilme", "idelenco", "titulopincipal", "titulooriginal",
                     "anolancamento", "numerovotos", "notamedia")
filme.show(5)

filme.createOrReplaceTempView("filme")
artista.createOrReplaceTempView("artista")
categoriaidade.createOrReplaceTempView("categoriaidade")
personagem.createOrReplaceTempView("personagem")
genero.createOrReplaceTempView("genero")
filmegenero.createOrReplaceTempView("filmegenero")

# tabelas dimensoes
# Supondo que df seja seu DataFrame com dados da tabela Filme
dimfilme = df.select(
    'idfilme', 'titulopincipal', 'titulooriginal', 'anolancamento',
    'numerovotos', 'notamedia'
).distinct()
dimfilme.createOrReplaceTempView("dimfilme")

# Unir as tabelas com aliases
dimelencopersonagem = elencopersonagem.alias("ep") \
    .join(artista.alias("a"), col("ep.idartista") == col("a.idartista"), "left") \
    .join(categoriaidade.alias("c"), col("c.idcategoriaidade") == col("a.idcategoriaidade"), "left") \
    .join(personagem.alias("p"), col("ep.idpersonagem") == col("p.idpersonagem"), "left") \
    .select(
        col("ep.idelenco"),
        col("ep.idpersonagem"),
        col("ep.idartista"),
        col("a.nomeartista"),
        col("a.nomeoriginal"),
        col("a.idadeartista"),
        col("a.generoartista"),
        col("a.anonascimento"),
        col("a.avaliacaoartista"),
        col("p.personagem"),
        col('c.idcategoriaidade'),
        col('c.categoriaIdade')
    ) \
    .distinct()
dimelencopersonagem.createOrReplaceTempView("dimelencopersonagem")

dimgenero = filmegenero.alias('fg') \
    .join(
        genero.alias('g'),
        coalesce(col('fg.idgenero'), lit(-1)) == coalesce(col('g.idgenero'), lit(-1)),
        'inner'
    ).select(
        col('fg.idgenero'),
        col('g.generofilme'),
        col('fg.idfilme')
    ).distinct()
dimgenero.createOrReplaceTempView("dimgenero")

dimfilme.show(5)
dimelencopersonagem.show(5)
dimgenero.show(5)

df.createOrReplaceTempView("df")

# Unir as tabelas para criar a tabela fato
fatofilme = filme.alias("f") \
    .join(dimfilme.alias("d"), col("f.idfilme") == col("d.idfilme"), "left") \
    .join(dimelencopersonagem.alias("e"), col("f.idelenco") == col("e.idelenco"), "left") \
    .join(dimgenero.alias("g"), col("f.idfilme") == col("g.idfilme"), "left") \
    .select(
        col("d.idfilme"),
        col("e.idelenco"),
        col("g.idgenero"),
    ) \
    .distinct()

# tabela temporária para a tabela fato
fatofilme.createOrReplaceTempView("fatofilme")
fatofilme.show(5)

# Salvando as dimensões no Glue Catalog
save_to_glue_catalog(filme, output_path + 'filme/', "modelagem-dimensoes", "filme")
save_to_glue_catalog(artista, output_path + 'artista/', "modelagem-dimensoes", "artista")
save_to_glue_catalog(categoriaidade, output_path + 'categoriaidade/', "modelagem-dimensoes", "categoriaidade")
save_to_glue_catalog(personagem, output_path + 'personagem/', "modelagem-dimensoes", "personagem")
save_to_glue_catalog(genero, output_path + 'genero/', "modelagem-dimensoes", "genero")
save_to_glue_catalog(filmegenero, output_path + 'filmegenero/', "modelagem-dimensoes", "filmegenero")

save_to_glue_catalog(dimfilme, output_path + 'dimfilme/', "modelagem-dimensoes", "dimfilme")
save_to_glue_catalog(dimelencopersonagem, output_path + 'dimelencopersonagem/', "modelagem-dimensoes", "dimelencopersonagem")
save_to_glue_catalog(dimgenero, output_path + 'dimgenero/', "modelagem-dimensoes", "dimgenero")
save_to_glue_catalog(fatofilme, output_path + 'fatofilme/', "modelagem-dimensoes", "fatofilme")

job.commit()

## Views

In [28]:
# Qual a quantidade de filmes realizados em cada ano?
# Combinando as consultas
quantidade_e_media_nota_por_ano = spark.sql("""
    SELECT
        anolancamento,
        COUNT(idfilme) AS quantidade_filmes,
        ROUND(AVG(notamedia),2) AS media_nota
    FROM
        DimFilme
    GROUP BY
        anolancamento
    ORDER BY
        anolancamento
""")
quantidade_e_media_nota_por_ano.show()

+-------------+-----------------+----------+
|anolancamento|quantidade_filmes|media_nota|
+-------------+-----------------+----------+
|         2000|                5|       5.6|
|         2001|                3|      6.33|
|         2002|                4|      5.75|
|         2003|                2|       7.0|
|         2004|                4|      6.25|
|         2005|                4|      6.25|
|         2006|                6|      5.67|
|         2007|                3|      6.67|
|         2008|                5|       5.6|
|         2009|                4|       6.0|
|         2010|                4|       5.5|
|         2011|                5|       5.4|
|         2012|                8|      5.63|
|         2013|                6|      5.83|
|         2014|                3|      5.67|
|         2015|               10|       5.4|
|         2016|                6|      5.67|
|         2017|                9|      5.44|
|         2018|                9|      5.11|
|         

In [114]:
# Executar a consulta SQL
artistas_avaliados_query = """
WITH AvaliacoesArtistas AS (
    SELECT
        de.idelenco,
        de.idcategoriaidade,
        de.nomeartista,
        de.categoriaIdade,
        de.avaliacaoartista,
        df.idfilme,
        df.titulopincipal AS filme,
        df.notamedia AS nota_filme,
        ROW_NUMBER() OVER (PARTITION BY de.nomeartista ORDER BY df.notamedia DESC, df.idfilme) AS ranking_filme
    FROM
        DimElencoPersonagem de
    JOIN
        FatoFilme ff ON de.idelenco = ff.idelenco
    JOIN
        DimFilme df ON ff.idfilme = df.idfilme
    JOIN
        DimGenero dg ON ff.idgenero = dg.idgenero
    WHERE
        dg.generofilme IN ('War', 'Crime')
),
ArtistasBemAvaliados AS (
    SELECT
        idcategoriaidade,
        nomeartista,
        categoriaIdade,
        MAX(avaliacaoartista) AS max_avaliacao
    FROM
        DimElencoPersonagem
    GROUP BY
        idcategoriaidade,
        nomeartista,
        categoriaIdade
)
SELECT
    aa.nomeartista,
    aa.categoriaIdade,
    aa.filme,
    aa.nota_filme
FROM
    AvaliacoesArtistas aa
JOIN
    ArtistasBemAvaliados aba ON aa.nomeartista = aba.nomeartista AND aa.avaliacaoartista = aba.max_avaliacao
WHERE
    aa.ranking_filme = 1
ORDER BY
    aa.categoriaIdade, aa.nota_filme DESC;
"""

artistas_avaliados = spark.sql(artistas_avaliados_query)
artistas_avaliados.show(200)

+--------------------+--------------+--------------------+----------+
|         nomeartista|categoriaIdade|               filme|nota_filme|
+--------------------+--------------+--------------------+----------+
|        Ana de Armas|       adultos|          Knives Out|         7|
|  Scarlett Johansson|       adultos|         Jojo Rabbit|         7|
|           Zac Efron|       adultos|Extremely Wicked,...|         6|
|         Amber Heard|       adultos|      The Stepfather|         5|
|           Joey King|       adultos|            Borealis|         5|
|     Mackenzie Davis|       adultos|      Bad Turn Worse|         5|
|      Sydney Sweeney|       adultos|Stolen from Suburbia|         5|
|     Melissa Benoist|       adultos|           Billy Boy|         4|
|         Johnny Depp|        idosos|                Blow|         7|
|    Sigourney Weaver|        idosos|            Infamous|         7|
|  Millie Bobby Brown|       juvenil|      Enola Holmes 2|         6|
|       Jason Statha

In [89]:
filmes_categoriaidade_qtdartistas_avgnota = (
    """
    WITH FilmePorCategoria AS (
    SELECT
        de.idcategoriaidade,
        de.categoriaIdade,
        COUNT(DISTINCT df.idfilme) AS quantidade_filmes,
        COUNT(DISTINCT de.idartista) AS quantidade_artistas,
        ROUND(AVG(df.notamedia),2) AS media_notas_filmes
    FROM
        DimElencoPersonagem de
    JOIN
        FatoFilme ff ON de.idelenco = ff.idelenco
    JOIN
        DimFilme df ON ff.idfilme = df.idfilme
    GROUP BY
        de.idcategoriaidade,
        de.categoriaIdade
)
SELECT
    categoriaIdade,
    quantidade_filmes,
    quantidade_artistas,
    media_notas_filmes
FROM
    FilmePorCategoria
ORDER BY
    quantidade_filmes DESC,
    quantidade_artistas DESC,
    media_notas_filmes DESC;
    """
)

catidade_artistas_avgnota = spark.sql(filmes_categoriaidade_qtdartistas_avgnota)
catidade_artistas_avgnota.show()

+--------------+-----------------+-------------------+------------------+
|categoriaIdade|quantidade_filmes|quantidade_artistas|media_notas_filmes|
+--------------+-----------------+-------------------+------------------+
|       maduros|               81|                 11|              5.91|
|       adultos|               26|                  8|              4.96|
|        idosos|               10|                  2|               6.1|
|       juvenil|                2|                  1|               6.0|
+--------------+-----------------+-------------------+------------------+



In [88]:
artista_nome_catidade_aval_medianota = (
    """
    WITH FilmesPorArtista AS (
    SELECT
        de.idartista,
        de.nomeartista,
        de.idcategoriaidade,
        de.categoriaIdade,
        de.avaliacaoartista,
        COUNT(DISTINCT df.idfilme) AS quantidade_filmes,
        ROUND(AVG(df.notamedia),2) AS media_notas_filmes
    FROM
        DimElencoPersonagem de
    JOIN
        FatoFilme ff ON de.idelenco = ff.idelenco
    JOIN
        DimFilme df ON ff.idfilme = df.idfilme
    GROUP BY
        de.idartista,
        de.nomeartista,
        de.idcategoriaidade,
        de.categoriaIdade,
        de.avaliacaoartista
)
SELECT
    nomeartista,
    categoriaIdade,
    avaliacaoartista,
    quantidade_filmes,
    media_notas_filmes
FROM
    FilmesPorArtista
ORDER BY
    avaliacaoartista DESC,
    quantidade_filmes DESC,
    media_notas_filmes DESC;
    """
)

FilmesPorArtista = spark.sql(artista_nome_catidade_aval_medianota)
FilmesPorArtista.show()

+--------------------+--------------+----------------+-----------------+------------------+
|         nomeartista|categoriaIdade|avaliacaoartista|quantidade_filmes|media_notas_filmes|
+--------------------+--------------+----------------+-----------------+------------------+
|  Scarlett Johansson|       adultos|          149.62|                6|               5.5|
|    Sigourney Weaver|        idosos|         144.936|                5|               5.8|
|         Johnny Depp|        idosos|         136.173|                5|               6.4|
|       Rosamund Pike|       maduros|         132.383|                4|              6.25|
|Benedict Cumberbatch|       maduros|          130.61|                3|               6.0|
|Joseph Gordon-Levitt|       maduros|         127.441|                7|              6.43|
|        Ana de Armas|       adultos|         117.635|                7|               5.0|
|        Ma Dong-seok|       maduros|         116.992|               15|        

In [51]:
filmes_qtdcategoria_medianotas = (
    """
    WITH FilmesPorGenero AS (
    SELECT
        dg.generofilme,
        COUNT(DISTINCT df.idfilme) AS quantidade_filmes,
        ROUND(AVG(df.notamedia),2) AS media_notas_filmes
    FROM
        DimGenero dg
    JOIN
        FatoFilme ff ON dg.idgenero = ff.idgenero
    JOIN
        DimFilme df ON ff.idfilme = df.idfilme
    WHERE
        dg.generofilme IN ('War', 'Crime')
    GROUP BY
        dg.generofilme
)
SELECT
    generofilme,
    quantidade_filmes,
    media_notas_filmes
FROM
    FilmesPorGenero
ORDER BY
    generofilme;
    """
)

FilmesPorGenero = spark.sql(filmes_qtdcategoria_medianotas)
FilmesPorGenero.show()

+-----------+-----------------+------------------+
|generofilme|quantidade_filmes|media_notas_filmes|
+-----------+-----------------+------------------+
|      Crime|              113|               5.7|
|        War|                5|               6.2|
+-----------+-----------------+------------------+



In [83]:
filmes_top5_crime_nota_avalartista = (
    """
WITH AvaliacoesPorFilme AS (
    SELECT
        df.idfilme,
        df.titulopincipal AS filme,
        dg.generofilme,
        df.notamedia AS nota_filme,
        de.nomeartista,
        de.avaliacaoartista,
        ROW_NUMBER() OVER (
            PARTITION BY df.idfilme
            ORDER BY de.avaliacaoartista DESC
        ) AS ranking_artista
    FROM
        DimGenero dg
    JOIN
        FatoFilme ff ON dg.idgenero = ff.idgenero
    JOIN
        DimFilme df ON ff.idfilme = df.idfilme
    JOIN
        DimElencoPersonagem de ON de.idelenco = ff.idelenco
    WHERE
        dg.generofilme = 'Crime'
)
SELECT
    filme,
    nota_filme,
    nomeartista,
    avaliacaoartista
FROM
    AvaliacoesPorFilme
WHERE
    ranking_artista = 1
ORDER BY
    nota_filme DESC,
    avaliacaoartista DESC
LIMIT 5;
    """
)

AvaliacoesPorFilme = spark.sql(filmes_top5_crime_nota_avalartista)
AvaliacoesPorFilme.show(300)

+--------------+----------+----------------+----------------+
|         filme|nota_filme|     nomeartista|avaliacaoartista|
+--------------+----------+----------------+----------------+
|      Dogville|         8|    Paul Bettany|         111.077|
|        Snatch|         8|   Jason Statham|         110.092|
|      Infamous|         7|Sigourney Weaver|         144.936|
|          Blow|         7|     Johnny Depp|         136.173|
|Public Enemies|         7|     Johnny Depp|         136.173|
+--------------+----------+----------------+----------------+



In [84]:
filmes_top5_war_nota_avalartista = (
    """
WITH AvaliacoesPorFilme AS (
    SELECT
        df.idfilme,
        df.titulopincipal AS filme,
        dg.generofilme,
        df.notamedia AS nota_filme,
        de.nomeartista,
        de.avaliacaoartista,
        ROW_NUMBER() OVER (
            PARTITION BY df.idfilme
            ORDER BY de.avaliacaoartista DESC
        ) AS ranking_artista
    FROM
        DimGenero dg
    JOIN
        FatoFilme ff ON dg.idgenero = ff.idgenero
    JOIN
        DimFilme df ON ff.idfilme = df.idfilme
    JOIN
        DimElencoPersonagem de ON de.idelenco = ff.idelenco
    WHERE
        dg.generofilme = 'War'
)
SELECT
    filme,
    nota_filme,
    nomeartista,
    avaliacaoartista
FROM
    AvaliacoesPorFilme
WHERE
    ranking_artista = 1
ORDER BY
    nota_filme DESC,
    avaliacaoartista DESC
LIMIT 5;
    """
)

AvaliacoesPorFilme = spark.sql(filmes_top5_war_nota_avalartista)
AvaliacoesPorFilme.show(300)

+-------------+----------+--------------------+----------------+
|        filme|nota_filme|         nomeartista|avaliacaoartista|
+-------------+----------+--------------------+----------------+
|  Jojo Rabbit|         7|  Scarlett Johansson|          149.62|
|A Private War|         6|       Rosamund Pike|         132.383|
|   Naples '44|         6|Benedict Cumberbatch|          130.61|
|    Stop-Loss|         6|Joseph Gordon-Levitt|         127.441|
|Journey's End|         6|        Paul Bettany|         111.077|
+-------------+----------+--------------------+----------------+

